In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import json
from pathlib import Path

base_path1 = "/capstor/store/cscs/swissai/a0142/agents_uq/lcb_llm_tool_agent_gpt_oss_20b/2368761_20260524_153248/" #lcb-hard
base_path2 = "/capstor/store/cscs/swissai/a0142/agents_uq/lcb_llm_tool_agent_gpt_oss_20b/2368762_20260524_153248/" #lcb-medium

base_path1 = "/capstor/store/cscs/swissai/a0142/agents_uq/lcb_llm_tool_agent_gpt_oss_20b/2644566_20260629_140906/readable/lcb_hard/"
base_path2 = "/capstor/store/cscs/swissai/a0142/agents_uq/lcb_llm_tool_agent_gpt_oss_20b/2644566_20260629_140906/readable/lcb_medium/"

results = pd.read_csv(base_path2 + "final_logprob_bayes_quality.csv")
results_traj = pd.read_csv(base_path2 + "generation_trajectory_scores.csv")
tool_success = pd.read_csv(base_path2 + "tool_success_by_instance.csv")   

summary_path = Path(base_path2) / "analysis_summary.json"
prior = json.loads(summary_path.read_text())["prior"]["prior_Y1"]

results = results[results["split"] == "test"].copy()
results_traj = results_traj[results_traj["split"] == "test"].copy()

results["tool_success"] = (
    tool_success["tool_no_final_before_verify_pass_success_rate"]
    .fillna(prior)
)

verb = pd.DataFrame(json.loads(l) for l in open("/capstor/store/cscs/swissai/a0142/agents_uq/lcb_llm_tool_agent_gpt_oss_20b/2644566_20260629_140906/lcb_medium__gpt_oss_20b_local.verbalized_2s.jsonl"))
verb["instance_id"] = verb["instance_id"].astype(str)
results = results.assign(instance_id=results["instance_id"].astype(str)).merge(verb[["instance_id", "verbalized_2s_confidence", "verbalized_2s_uncertainty"]], on="instance_id", how="left")

In [22]:
from lm_polygraph.ue_metrics import PredictionRejectionArea
from lm_polygraph.ue_metrics.ue_metric import (
    get_random_scores,
    normalize_metric,
)
from sklearn.metrics import roc_auc_score

prrs = {
        #"PRR": PredictionRejectionArea(), 
        "PRR_05": PredictionRejectionArea(max_rejection=0.5), 
        # "Spearmanr": stats.spearmanr,
        # "AUROC": lambda y_true, y_score: roc_auc_score(y_true, y_score),
        }

In [26]:
df = {}
for prr in prrs:
    df[prr] = {}
    for col in ["llm_perplexity", "llm_log_seq_prob", "verbalized_2s_confidence_x", "tool_success", "bayes_state"]:
        if col == "quality":
            continue
        if prr == "Spearmanr":
            normalized_score = prrs[prr](results[col], results["quality"]).statistic
        elif prr == "AUROC":
            normalized_score = prrs[prr](results["quality"].values, results[col].values)
        else:
            ue_metric = prrs[prr](-results[col].values, results["quality"].values)
            oracle_score = prrs[prr](-results["quality"].values, results["quality"].values)
            random_score = get_random_scores(prrs[prr], results["quality"])
            normalized_score = normalize_metric(ue_metric, oracle_score, random_score)
        df[prr][col] = normalized_score
        print("End-to-end {} for {}: {}".format(prr, col, normalized_score))
df = pd.DataFrame(df)
df.style.background_gradient()

End-to-end PRR_05 for llm_perplexity: 0.2508781040844699
End-to-end PRR_05 for llm_log_seq_prob: 0.6273048320803996
End-to-end PRR_05 for verbalized_2s_confidence_x: 0.459855317952976
End-to-end PRR_05 for tool_success: 0.872513203304385
End-to-end PRR_05 for bayes_state: 0.8682057140546424


,PRR_05
llm_perplexity,0.250878
llm_log_seq_prob,0.627305
verbalized_2s_confidence_x,0.459855
tool_success,0.872513
bayes_state,0.868206


In [28]:
base_path1 = "/capstor/store/cscs/swissai/a0142/agents_uq/lcb_llm_tool_agent_gpt_oss_20b/2368761_20260524_153248/" #lcb-hard
base_path1 = "/capstor/store/cscs/swissai/a0142/agents_uq/lcb_llm_tool_agent_gpt_oss_20b/2644566_20260629_140906/readable/lcb_hard/"

results = pd.read_csv(base_path1 + "final_logprob_bayes_quality.csv")
results_traj = pd.read_csv(base_path1 + "generation_trajectory_scores.csv")
tool_success = pd.read_csv(base_path1 + "tool_success_by_instance.csv")   

summary_path = Path(base_path1) / "analysis_summary.json"
prior = json.loads(summary_path.read_text())["prior"]["prior_Y1"]

results = results[results["split"] == "test"].copy()
results_traj = results_traj[results_traj["split"] == "test"].copy()

results["tool_success"] = (
    tool_success["tool_no_final_before_verify_pass_success_rate"]
    .fillna(prior)
)

verb = pd.DataFrame(json.loads(l) for l in open("/capstor/store/cscs/swissai/a0142/agents_uq/lcb_llm_tool_agent_gpt_oss_20b/2644566_20260629_140906/lcb_hard__gpt_oss_20b_local.verbalized_2s.jsonl"))
verb["instance_id"] = verb["instance_id"].astype(str)
results = results.assign(instance_id=results["instance_id"].astype(str)).merge(verb[["instance_id", "verbalized_2s_confidence", "verbalized_2s_uncertainty"]], on="instance_id", how="left")

In [30]:
df1 = {}
for prr in prrs:
    df1[prr] = {}
    for col in ["llm_perplexity", "llm_log_seq_prob", "verbalized_2s_confidence_x", "tool_success", "bayes_state"]:
        if col == "quality":
            continue
        if prr == "Spearmanr":
            normalized_score = prrs[prr](results[col], results["quality"]).statistic
        elif prr == "AUROC":
            normalized_score = prrs[prr](results["quality"].values, results[col].values)
        else:
            ue_metric = prrs[prr](-results[col].values, results["quality"].values)
            oracle_score = prrs[prr](-results["quality"].values, results["quality"].values)
            random_score = get_random_scores(prrs[prr], results["quality"])
            normalized_score = normalize_metric(ue_metric, oracle_score, random_score)
        df1[prr][col] = normalized_score
        print("End-to-end {} for {}: {}".format(prr, col, normalized_score))
df1 = pd.DataFrame(df1)
df1.style.background_gradient()

End-to-end PRR_05 for llm_perplexity: 0.37446272255766844
End-to-end PRR_05 for llm_log_seq_prob: 0.6803018363980909
End-to-end PRR_05 for verbalized_2s_confidence_x: 0.3155958336530689
End-to-end PRR_05 for tool_success: 0.5112013131219091
End-to-end PRR_05 for bayes_state: 0.8053251552226337


,PRR_05
llm_perplexity,0.374463
llm_log_seq_prob,0.680302
verbalized_2s_confidence_x,0.315596
tool_success,0.511201
bayes_state,0.805325


In [31]:
df_final = pd.concat([df, df1], axis=1, keys=["lcb-medium", "lcb-hard"])
df_final.style.background_gradient()

,lcb-medium,lcb-hard
,PRR_05,PRR_05
llm_perplexity,0.250878,0.374463
llm_log_seq_prob,0.627305,0.680302
verbalized_2s_confidence_x,0.459855,0.315596
tool_success,0.872513,0.511201
bayes_state,0.868206,0.805325


In [32]:
df_final.columns = ["LCB-Medium", "LCB-Hard"]
df_final.index = ["Perplexity", "Seq. Prob.", "Verb.", "Tool Success Rate", "Bayes Belief State"]

In [33]:
df_final["Average"] = df_final.mean(axis=1)
df_final.style.background_gradient()

,LCB-Medium,LCB-Hard,Average
Perplexity,0.250878,0.374463,0.312670
Seq. Prob.,0.627305,0.680302,0.653803
Verb.,0.459855,0.315596,0.387726
Tool Success Rate,0.872513,0.511201,0.691857
Bayes Belief State,0.868206,0.805325,0.836765


In [34]:
print(df_final.round(3).to_latex())

\begin{tabular}{lrrr}
\toprule
 & LCB-Medium & LCB-Hard & Average \\
\midrule
Perplexity & 0.251000 & 0.374000 & 0.313000 \\
Seq. Prob. & 0.627000 & 0.680000 & 0.654000 \\
Verb. & 0.460000 & 0.316000 & 0.388000 \\
Tool Success Rate & 0.873000 & 0.511000 & 0.692000 \\
Bayes Belief State & 0.868000 & 0.805000 & 0.837000 \\
\bottomrule
\end{tabular}

